# Class Disparities in Texas Traffic Stops

In this notebook, we will work through a simplified version of the analysis from a recent economics paper "Class Disparities and Discrimination in Traffic Stops and Searches".

The main question is:

**How does police search behavior vary with a motorist's economic status?**

We will do five things:

1. Look at the structure of the stop data.
2. Discuss how the paper imputes household income.
3. See how the main analysis sample is constructed.
4. Reproduce two descriptive figures:
   - search rates vs. household income
   - hit rates vs. household income
5. Build a within-motorist dataset and make a plot based on repeated stops of the same motorist.

The two data files we will use are:

- `stops_all.dta`: a (close to) raw stop-level file
- `all_stops_clean.dta`: the final analysis sample used by most of the paper

While some of the raw data we use in the paper is restricted access and not publicly available, you can find raw data with significant overlap to what we use here at these sources:

- Stanford Open Policing Project: https://openpolicing.stanford.edu/
- Texas Department of Public Safety website: https://www.dps.texas.gov/section/highway-patrol/texas-highway-patrol-high-value-data-sets

In [ ]:
# Import the libraries we will use throughout the notebook
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

# Set paths
repo = Path.cwd()
data_dir = repo / "data" / "intermediate"
output_dir = repo / "output" / "lecture_notebook_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

stops_all_path = data_dir / "stops_all.dta"
all_stops_clean_path = data_dir / "all_stops_clean.dta"

## 1. Looking at the stop data

We begin with `stops_all.dta`, which close to (but not quite) the raw CSV files from the Texas Department of Public Safety.

Let's take a look at some of the key columns.

In [ ]:
# Load a few columns from the stop file so we can inspect the structure
raw_columns = [
    "ha_arrest_key",
    "ha_arrest_date",
    "ha_race_sex",
    "ha_a_state_drvr",
    "ha_veh_make",
    "ha_vehicle_type",
    "ha_veh_year",
    "ha_officer_id",
    "ha_latitude",
    "ha_longitude",
    "ha_reason_cita",
    "ha_reason_warn",
    "ha_searched",
    "ha_contraban",
]

stops_preview = next(
    pd.read_stata(
        stops_all_path,
        columns=raw_columns,
        convert_categoricals=False,
        chunksize=5,
    )
)

stops_preview

## 2. How income is imputed

We do **not** observe household income directly in the stop records. Instead, we impute income using the motorist's address.

The rough idea is:

1. Use the driver's exact address to locate the residence.
2. Use Census block-group income distributions from the American Community Survey (ACS).
3. If the motorist lives in a single-family home, use the property's assessed value to rank that residence within the block group.
4. Map that property-value percentile to a percentile of the homeowner income distribution in the same block group.
5. If the motorist lives in multifamily housing or cannot be matched to a specific property, use the median renter income category for the block group.
6. Convert the resulting income category into an imputed log income value.

The property value data are restricted access, so we will not rebuild this step here. But we can illustrate the logic with a toy example.

In [ ]:
# This is a toy example that illustrates the logic of the imputation step. (These are not real records from the data.)
toy_homes = pd.DataFrame(
    {
        "address": ["A", "B", "C", "D"],
        "block_group": ["BG1", "BG1", "BG1", "BG1"],
        "property_value": [180000, 260000, 420000, 700000],
    }
)

# Rank each property within its block group
toy_homes["property_percentile"] = toy_homes.groupby("block_group")["property_value"].rank(pct=True)

# Suppose we know the homeowner income distribution for that block group
# We can then map each percentile to an income category
toy_homes["imputed_income_bin"] = pd.cut(
    toy_homes["property_percentile"],
    bins=[0, 0.25, 0.5, 0.75, 1.0],
    labels=["25-35k", "35-50k", "50-75k", "100k+"],
)

toy_homes

Once those income categories are assigned, the cleaned dataset `all_stops_clean.dta` contains the final imputed income variable `ln_income_imp`.

## 3. How the analysis sample is constructed

The final analysis file is `all_stops_clean.dta`.

The paper starts with a much larger set of stops and then applies a series of restrictions. The main counts are:

| Step | Remaining stops |
| --- | ---: |
| Unique stops after deduplication | 15,761,299 |
| Drop missing officer / race / stop outcome | 15,758,979 |
| Keep Texas drivers | 13,886,751 |
| Keep valid address and income | 11,928,370 |
| Keep passenger cars, pickups, and SUVs with valid vehicle status | 11,350,965 |
| Drop missing time or location bins | 11,021,719 |
| Drop toll violations | 11,006,885 |

Now let's look at a few columns from the cleaned analysis sample.

In [ ]:
# Load a few columns from the cleaned analysis file
clean_columns = [
    "id",
    "stop_time",
    "ln_income_imp",
    "inc_cat",
    "ln_vehinc",
    "ha_searched",
    "ha_contraban",
    "veh_make",
    "veh_type",
    "veh_age",
    "speeding",
    "dwi",
    "moving"
]

clean_preview = next(
    pd.read_stata(
        all_stops_clean_path,
        columns=clean_columns,
        convert_categoricals=False,
        chunksize=5,
    )
)

clean_preview

## 4. Summary statistics

We will load only the columns we need and compute a few summary statistics from the cleaned sample.

In [ ]:
# Read a small set of variables from the cleaned file
summary_columns = [
    "ln_income_imp",
    "ha_searched",
    "ha_contraban",
    "speeding",
    "dwi",
    "moving",
]

summary = pd.read_stata(
    all_stops_clean_path,
    columns=summary_columns,
    convert_categoricals=False,
)

In [ ]:
# Build a readable table
summary_table = pd.DataFrame(
    {
        "Statistic": [
            "Number of stops",
            "Search rate (%)",
            "Unconditional contraband rate (%)",
            "Hit rate, conditional on search (%)",
            "Mean log household income",
            "Share speeding (%)",
            "Share DWI (%)",
            "Share moving violation (%)",
        ],
        "Value": [
            f"{len(summary):,}",
            f"{100 * summary['ha_searched'].mean():.2f}",
            f"{100 * summary['ha_contraban'].mean():.2f}",
            f"{100 * summary.loc[summary['ha_searched'] == 1, 'ha_contraban'].mean():.2f}",
            f"{summary['ln_income_imp'].mean():.2f}",
            f"{100 * summary['speeding'].mean():.2f}",
            f"{100 * summary['dwi'].mean():.2f}",
            f"{100 * summary['moving'].mean():.2f}",
        ],
    }
)

summary_table

## 5. Understanding the vehicle status measure

The paper uses a variable called `ln_vehinc`, which is a measure of predicted household income based on the vehicle involved in a stop.

Two important inputs are:

- the **make** of the vehicle
- the **age** of the vehicle

To make that idea concrete, the next figure compares an economy brand and a luxury brand. Here we use **Honda** and **BMW**, and we focus on passenger cars.

In [ ]:
# Load just the columns needed for the vehicle-status illustration
vehicle_status = pd.read_stata(
    all_stops_clean_path,
    columns=["veh_make", "veh_type", "veh_age", "ln_vehinc"],
    convert_categoricals=False,
)

# Keep Honda and BMW passenger cars with ages in a reasonable range
vehicle_status = vehicle_status.loc[
    (vehicle_status["veh_make"].isin(["HONDA", "BMW"]))
    & (vehicle_status["veh_type"] == 1)
    & (vehicle_status["veh_age"] >= 0)
    & (vehicle_status["veh_age"] <= 20)
]

# Average ln_vehinc by make and age
status_by_age = (
    vehicle_status
    .groupby(["veh_make", "veh_age"])
    .agg(avg_ln_vehinc=("ln_vehinc", "mean"))
    .reset_index()
)

status_by_age

In [ ]:
#Make plot
fig, ax = plt.subplots(figsize=(8, 5))

for make, color in [("HONDA", "steelblue"), ("BMW", "darkred")]:
    temp = status_by_age.loc[status_by_age["veh_make"] == make]
    ax.plot(temp["veh_age"], temp["avg_ln_vehinc"], marker="o", linewidth=2, label=make, color=color)

ax.set_xlabel("Vehicle age")
ax.set_ylabel("Predicted log household income given vehicle (ln_vehinc)")
ax.set_title("Vehicle status depends on make and age")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Search rates vs. household income

The paper groups stops into 16 household income bins. For each bin, it plots the average search rate.

We will reproduce that logic here.

In [ ]:
# Load the columns needed for the search-rate figure
search_data = pd.read_stata(
    all_stops_clean_path,
    columns=["inc_cat", "ln_income_imp", "ha_searched"],
    convert_categoricals=False,
)

search_data = search_data.dropna()

# Compute the average search rate within each income bin
search_by_income = (
    search_data
    .groupby("inc_cat")
    .agg(
        mean_log_income=("ln_income_imp", "mean"),
        search_rate=("ha_searched", "mean"),
    )
    .reset_index()
)

# Re-scale as percentage
search_by_income["search_rate"] = 100 * search_by_income["search_rate"]

income_labels = {
    1: "<10", 2: "10-15", 3: "15-20", 4: "20-25",
    5: "25-30", 6: "30-35", 7: "35-40", 8: "40-45",
    9: "45-50", 10: "50-60", 11: "60-75", 12: "75-100",
    13: "100-125", 14: "125-150", 15: "150-200", 16: ">200",
}

# Estimate a simple bivariate regression to get the slope
X = sm.add_constant(search_data["ln_income_imp"])
search_model = sm.OLS(100 * search_data["ha_searched"], X).fit()

search_by_income

In [ ]:
# Make the plot
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(search_by_income["mean_log_income"], search_by_income["search_rate"], color="maroon", s=45)
ax.set_xticks(search_by_income["mean_log_income"])
ax.set_xticklabels([income_labels[i] for i in search_by_income["inc_cat"]], rotation=45, ha="right")
ax.set_xlabel("Household income bin (thousands of dollars)")
ax.set_ylabel("Search rate (%)")
ax.set_title("Search rates fall as household income rises")
ax.text(
    0.98,
    0.95,
    f"Slope = {search_model.params['ln_income_imp']:.2f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    bbox=dict(facecolor="white", edgecolor="0.8"),
)
plt.tight_layout()
plt.show()

## 7. Hit rates vs. household income

A **hit rate** is the percentage of searches that recover contraband (e.g. drugs, weapons).

To look at hit rates, we restrict attention to stops that actually involved a search.

In [ ]:
# Load the columns needed for the hit rate figure
hit_data = pd.read_stata(
    all_stops_clean_path,
    columns=["inc_cat", "ln_income_imp", "ha_searched", "ha_contraban"],
    convert_categoricals=False,
)

# Keep only searches
hit_data = hit_data.loc[hit_data["ha_searched"] == 1].dropna()

# Compute hit rates within income bins
hit_by_income = (
    hit_data
    .groupby("inc_cat")
    .agg(
        mean_log_income=("ln_income_imp", "mean"),
        hit_rate=("ha_contraban", "mean"),
    )
    .reset_index()
)

hit_by_income["hit_rate"] = 100 * hit_by_income["hit_rate"]

# Estimate a simple bivariate regression to get the slope
X = sm.add_constant(hit_data["ln_income_imp"])
hit_model = sm.OLS(100 * hit_data["ha_contraban"], X).fit()

hit_by_income

In [ ]:
# Make the plot
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(hit_by_income["mean_log_income"], hit_by_income["hit_rate"], color="maroon", s=45)
ax.set_xticks(hit_by_income["mean_log_income"])
ax.set_xticklabels([income_labels[i] for i in hit_by_income["inc_cat"]], rotation=45, ha="right")
ax.set_xlabel("Household income bin (thousands of dollars)")
ax.set_ylabel("Hit rate (%)")
ax.set_title("Hit rates rise as household income rises")
ax.text(
    0.98,
    0.05,
    f"Slope = {hit_model.params['ln_income_imp']:.2f}",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    bbox=dict(facecolor="white", edgecolor="0.8"),
)
plt.tight_layout()
plt.show()

## 8. Building a within-motorist dataset

Next, we want to compare repeated stops of the **same motorist**. Here we will focus on what happens when the same motorist is stopped multiple times, but their vehicle conveys different class signals across stops. The goal is to measure the effect of the class signal on police behavior; in other words, **class discrimination**.

To do that, we need to:

1. sort each driver's stops over time,
2. look at the previous stop for that same driver,
3. compare vehicle status and search outcomes across the two stops.

The key variables we want are:

- `same_car`: whether the same vehicle appears in consecutive stops
- `vehinc_diff`: change in vehicle status
- `search_diff`: change in search status, measured in percentage points

In [ ]:
# Load the variables we need to construct the within-motorist comparisons
within = pd.read_stata(
    all_stops_clean_path,
    columns=["id", "stop_time", "veh_make", "veh_type", "ha_veh_year", "ln_vehinc", "ha_searched"],
    convert_categoricals=False,
)

# Drop rows with missing values in the variables we need
within = within.dropna(subset=["id", "stop_time", "veh_make", "veh_type", "ha_veh_year", "ln_vehinc", "ha_searched"])

# Sort stops within each motorist
within = within.sort_values(["id", "stop_time"])

# Use shift(1) to look at the previous stop for the same motorist
within["prev_stop_time"] = within.groupby("id")["stop_time"].shift(1)
within["prev_veh_make"] = within.groupby("id")["veh_make"].shift(1)
within["prev_veh_type"] = within.groupby("id")["veh_type"].shift(1)
within["prev_ha_veh_year"] = within.groupby("id")["ha_veh_year"].shift(1)
within["prev_ln_vehinc"] = within.groupby("id")["ln_vehinc"].shift(1)
within["prev_search"] = within.groupby("id")["ha_searched"].shift(1)

# Keep only stops that have a previous stop
within = within.dropna(subset=["prev_ln_vehinc", "prev_search"])

# Check whether the same vehicle appears in both stops
within["same_car"] = (
    (within["veh_make"] == within["prev_veh_make"])
    & (within["veh_type"] == within["prev_veh_type"])
    & (within["ha_veh_year"] == within["prev_ha_veh_year"])
)

# Compute changes between the current stop and the previous stop
within["vehinc_diff"] = within["ln_vehinc"] - within["prev_ln_vehinc"]
within["search_diff"] = 100 * (within["ha_searched"] - within["prev_search"])

within[["id", "prev_stop_time", "stop_time", "same_car", "vehinc_diff", "search_diff"]].head()

## 9. A within-motorist plot

Now we group the changes in vehicle status into bins and plot the average change in search rates within each bin.

The hollow marker shows pairs of consecutive stops where the same vehicle appears in both stops. The filled markers show pairs where the vehicle changes.

For readability, we zoom in on the range from about -0.5 to 0.5 on both axes.

In [ ]:
# Split the data into same-vehicle and different-vehicle stop pairs
switched = within.loc[within["same_car"] == False]
same_vehicle = within.loc[within["same_car"] == True]

# Put the different-vehicle observations into 20 bins based on the change in vehicle status
switched["ventile"] = pd.qcut(switched["vehinc_diff"], 20, labels=False, duplicates="drop")

# Compute average x and y values within each bin
binned = (
    switched
    .groupby("ventile")
    .agg(
        mean_vehinc_diff=("vehinc_diff", "mean"),
        mean_search_diff=("search_diff", "mean"),
    )
    .reset_index()
)

# Estimate the slope in the switched-vehicle sample
X = sm.add_constant(switched["vehinc_diff"])
within_model = sm.OLS(switched["search_diff"], X).fit(
    cov_type="cluster",
    cov_kwds={"groups": switched["id"]},
)

# Create fitted values for the regression line
x_grid = np.linspace(-0.55, 0.55, 200)
y_grid = within_model.params["const"] + within_model.params["vehinc_diff"] * x_grid

# Average point for stop pairs with the same vehicle
same_x = same_vehicle["vehinc_diff"].mean()
same_y = same_vehicle["search_diff"].mean()

In [ ]:
# Make the plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(binned["mean_vehinc_diff"], binned["mean_search_diff"], color="navy", s=40, label="Different vehicle")
ax.plot(x_grid, y_grid, color="navy", linewidth=2)
ax.scatter(same_x, same_y, facecolors="none", edgecolors="navy", s=150, linewidth=2, label="Same vehicle")
ax.set_xlim(-0.55, 0.55)
ax.set_ylim(-0.55, 0.55)
ax.set_xlabel("Change in vehicle status")
ax.set_ylabel("Change in search rate (percentage points)")
ax.set_title("Search rates and changes in vehicle status")
ax.legend()
ax.text(
    0.98,
    0.95,
    f"Slope = {within_model.params['vehinc_diff']:.3f}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    bbox=dict(facecolor="white", edgecolor="0.8"),
)
plt.tight_layout()
plt.show()

## 10. Wrap-up

What do these descriptive patterns suggest?

- Search rates are higher for lower-income motorists.
- Hit rates are lower for lower-income motorists.
- Vehicle status captures an important visible signal of economic status.
- For the same motorist, they are more likely to be searched when they are stopped in a low-status vehicle.